In [1]:
import os
print(os.getcwd())

/home/yk/Coding/hrt-datathon/yehor


In [ ]:
ALPHA=40000
SENSITIVITY=1.0
CLIP=0.5


import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

DATA_DIR = "../data"

# ── 1. Load training data ─────────────────────────────────────────────────────

seen_train    = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"), engine="fastparquet")
unseen_train  = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")

# ── 2. Feature engineering ────────────────────────────────────────────────────

def build_features(bars: pd.DataFrame) -> pd.DataFrame:
    """Extract session-level features from seen OHLC bars."""

    def session_feats(grp):
        grp   = grp.sort_values("bar_ix")
        c     = grp["close"].values
        o     = grp["open"].values
        h     = grp["high"].values
        lo    = grp["low"].values
        n     = len(c)
        br    = np.diff(c) / c[:-1] if n > 1 else np.array([0.0])

        return pd.Series({
            "cum_return":   c[-1] / c[0] - 1,
            "last3_return": c[-1] / c[max(0, n - 4)] - 1,
            "slope":        np.polyfit(np.arange(n), c, 1)[0] / c.mean(),
            "realized_vol": np.std(br),
            "range_mean":   np.mean((h - lo) / c),
            "up_bar_frac":  np.mean(c > o),
            "wick_ratio":   np.mean((h - c) / (h - lo + 1e-8)),
        })

    return bars.groupby("session").apply(session_feats)

FEATURES = ["cum_return", "last3_return", "slope",
            "realized_vol", "range_mean", "up_bar_frac", "wick_ratio"]

# ── 3. Build training labels ──────────────────────────────────────────────────

mid_price = seen_train.groupby("session")["close"].last()
end_price = unseen_train.groupby("session")["close"].last()
y_train   = (end_price / mid_price - 1).rename("return")

X_train_raw = build_features(seen_train)[FEATURES]

# Align index (drop any sessions missing from either side)
idx = X_train_raw.index.intersection(y_train.index)
X_train_raw = X_train_raw.loc[idx]
y_train     = y_train.loc[idx]

# ── 4. Cross-validate to confirm we beat the baseline ────────────────────────

from sklearn.model_selection import KFold

def compute_sharpe(positions, returns):
    pnl = positions * returns
    return np.mean(pnl) / np.std(pnl) * 16

def make_positions(raw_preds, long_bias=1.0, sensitivity=0.3, clip=2.5):
    """
    Never go short (strong positive drift).
    Modulate position size around the long bias using model signal.
    position = long_bias + sensitivity * z_score(prediction)
    Clipped so minimum position is always > 0.
    """
    z = (raw_preds - raw_preds.mean()) / (raw_preds.std() + 1e-8)
    z = np.clip(z, -clip, clip)
    positions = long_bias + sensitivity * z
    positions = np.maximum(positions, 0.05)   # never fully flat
    return positions

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_raw)
y        = y_train.values

kf      = KFold(n_splits=5, shuffle=True, random_state=42)
cv_sharpes_model    = []
cv_sharpes_baseline = []

for train_idx, val_idx in kf.split(X_scaled):
    model = Ridge(alpha=ALPHA)
    model.fit(X_scaled[train_idx], y[train_idx])

    preds     = model.predict(X_scaled[val_idx])
    positions = make_positions(preds, sensitivity=SENSITIVITY, clip=CLIP)
    y_val     = y[val_idx]

    cv_sharpes_model.append(compute_sharpe(positions, y_val))
    cv_sharpes_baseline.append(compute_sharpe(np.ones(len(y_val)), y_val))

print(f"Baseline CV Sharpe : {np.mean(cv_sharpes_baseline):.4f} ± {np.std(cv_sharpes_baseline):.4f}")
print(f"Model    CV Sharpe : {np.mean(cv_sharpes_model):.4f} ± {np.std(cv_sharpes_model):.4f}")

# ── 5. Fit final model on all training data ───────────────────────────────────

final_model = Ridge(alpha=ALPHA)
final_scaler = StandardScaler()
X_final = final_scaler.fit_transform(X_train_raw)
final_model.fit(X_final, y)

print("\nFeature coefficients:")
for feat, coef in zip(FEATURES, final_model.coef_):
    print(f"  {feat:20s}  {coef:+.6f}")

# ── 6. Generate predictions for both test sets ────────────────────────────────

def predict_positions(bars: pd.DataFrame) -> pd.Series:
    feats    = build_features(bars)[FEATURES]
    X        = final_scaler.transform(feats)
    preds    = final_model.predict(X)
    positions = make_positions(preds)
    return pd.Series(positions, index=feats.index, name="target_position")

public_bars  = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"),  engine="fastparquet")
private_bars = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"), engine="fastparquet")

public_positions  = predict_positions(public_bars)
private_positions = predict_positions(private_bars)

# ── 7. Write submissions ──────────────────────────────────────────────────────

def write_submission(positions: pd.Series, path: str):
    sub = positions.reset_index()
    sub.columns = ["session", "target_position"]
    sub = sub.sort_values("session")
    sub.to_csv(path, index=False)
    print(f"Saved {len(sub)} rows → {path}")
    print(sub.describe())

write_submission(public_positions,  "submission_public.csv")
write_submission(private_positions, "submission_private.csv")

Baseline CV Sharpe : 2.7838 ± 0.5857
Model    CV Sharpe : 2.9824 ± 0.5270

Feature coefficients:
  cum_return            -0.000033
  last3_return          +0.000022
  slope                 -0.000032
  realized_vol          +0.000035
  range_mean            +0.000038
  up_bar_frac           -0.000028
  wick_ratio            +0.000033
Saved 10000 rows → submission_public.csv
           session  target_position
count  10000.00000     10000.000000
mean    5999.50000         0.999067
std     2886.89568         0.296346
min     1000.00000         0.250000
25%     3499.75000         0.790253
50%     5999.50000         0.983981
75%     8499.25000         1.200229
max    10999.00000         1.750000
Saved 10000 rows → submission_private.csv
           session  target_position
count  10000.00000     10000.000000
mean   15999.50000         0.999158
std     2886.89568         0.296497
min    11000.00000         0.250000
25%    13499.75000         0.794812
50%    15999.50000         0.985653
75%   

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

# ── 1. Configuration & Loading ────────────────────────────────────────────────
ALPHA = 10000
SENSITIVITY = 1.0
CLIP = 1.0

print("Loading data...")
# Load Price Data
seen_train    = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"), engine="fastparquet")
unseen_train  = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")
public_bars   = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"), engine="fastparquet")
private_bars  = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"), engine="fastparquet")

# Load Pre-processed Headline Data (assuming you saved it from the FinBERT step)
# Update these paths to match where your sentiment scores are saved!
headline_train   = pd.read_csv("headline_stats_seen_train.csv")
headline_public  = pd.read_csv("headline_stats_seen_pubtest.csv")
headline_private = pd.read_csv("headline_stats_seen_privtest.csv")

# ── 2. Feature Engineering (Price & Text) ─────────────────────────────────────

def build_price_features(bars: pd.DataFrame) -> pd.DataFrame:
    """Extract session-level features from OHLC bars."""
    def session_feats(grp):
        grp = grp.sort_values("bar_ix")
        c = grp["close"].values
        o = grp["open"].values
        h = grp["high"].values
        lo = grp["low"].values
        n = len(c)
        br = np.diff(c) / c[:-1] if n > 1 else np.array([0.0])

        return pd.Series({
            "cum_return":   c[-1] / c[0] - 1,
            "last3_return": c[-1] / c[max(0, n - 4)] - 1,
            "slope":        np.polyfit(np.arange(n), c, 1)[0] / c.mean(),
            "realized_vol": np.std(br),
            "range_mean":   np.mean((h - lo) / c),
            "up_bar_frac":  np.mean(c > o),
            "wick_ratio":   np.mean((h - c) / (h - lo + 1e-8)),
        })
    return bars.groupby("session").apply(session_feats)

def build_text_features(headlines: pd.DataFrame, bars: pd.DataFrame) -> pd.DataFrame:
    """Extract session-level features from headline sentiment and price convergence."""
    # Merge bars to get open/close for impact calculation
    merged = headlines.merge(
        bars[['session', 'bar_ix', 'open', 'close']], 
        on=['session', 'bar_ix'], 
        how='left'
    )
    
    # Calculate Impact (Realized Move * Sentiment Polarity)
    merged['bar_return'] = (merged['close'] - merged['open']) / (merged['open'] + 1e-8)
    merged['impact_score'] = merged['bar_return'].abs() * merged['polarity_score'].abs()
    
    def session_text_feats(grp):
        return pd.Series({
            "news_volume":    len(grp),
            "mean_sentiment": grp["linear_score"].mean(),
            "sum_sentiment":  grp["linear_score"].sum(),
            "mean_polarity":  grp["polarity_score"].mean(),
            "max_impact":     grp["impact_score"].max()
        })
        
    return merged.groupby("session").apply(session_text_feats)

def build_all_features(bars: pd.DataFrame, headlines: pd.DataFrame) -> pd.DataFrame:
    """Combines price and text features, filling 0 for sessions with no news."""
    X_price = build_price_features(bars)
    X_text = build_text_features(headlines, bars)
    # Left join ensures we keep all price sessions. fillna(0) handles zero-news sessions.
    return X_price.join(X_text, how='left').fillna(0)

# Feature Lists
PRICE_FEATURES = ["cum_return", "last3_return", "slope", "realized_vol", "range_mean", "up_bar_frac", "wick_ratio"]
TEXT_FEATURES  = ["news_volume", "mean_sentiment", "sum_sentiment", "mean_polarity", "max_impact"]
FEATURES = PRICE_FEATURES + TEXT_FEATURES

# ── 3. Build Training Labels & Matrix ─────────────────────────────────────────

print("Building features...")
mid_price = seen_train.groupby("session")["close"].last()
end_price = unseen_train.groupby("session")["close"].last()
y_train   = (end_price / mid_price - 1).rename("return")

X_train_raw = build_all_features(seen_train, headline_train)[FEATURES]

# Align index
idx = X_train_raw.index.intersection(y_train.index)
X_train_raw = X_train_raw.loc[idx]
y_train     = y_train.loc[idx]

# ── 4. Cross-Validation & Position Sizing ─────────────────────────────────────

def compute_sharpe(positions, returns):
    pnl = positions * returns
    return np.mean(pnl) / (np.std(pnl) + 1e-8) * 16

def make_positions(raw_preds, long_bias=1.0, sensitivity=0.3, clip=2.5):
    z = (raw_preds - raw_preds.mean()) / (raw_preds.std() + 1e-8)
    z = np.clip(z, -clip, clip)
    positions = long_bias + sensitivity * z
    positions = np.maximum(positions, 0.05)
    return positions

print("Running Cross-Validation...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_raw)
y = y_train.values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_sharpes_model = []
cv_sharpes_baseline = []

for train_idx, val_idx in kf.split(X_scaled):
    model = Ridge(alpha=ALPHA)
    model.fit(X_scaled[train_idx], y[train_idx])

    preds     = model.predict(X_scaled[val_idx])
    positions = make_positions(preds, sensitivity=SENSITIVITY, clip=CLIP)
    y_val     = y[val_idx]

    cv_sharpes_model.append(compute_sharpe(positions, y_val))
    cv_sharpes_baseline.append(compute_sharpe(np.ones(len(y_val)), y_val))

print(f"Baseline CV Sharpe : {np.mean(cv_sharpes_baseline):.4f} ± {np.std(cv_sharpes_baseline):.4f}")
print(f"Model    CV Sharpe : {np.mean(cv_sharpes_model):.4f} ± {np.std(cv_sharpes_model):.4f}")

# ── 5. Train Final Model ──────────────────────────────────────────────────────

print("\nTraining Final Model...")
final_model = Ridge(alpha=ALPHA)
final_scaler = StandardScaler()
X_final = final_scaler.fit_transform(X_train_raw)
final_model.fit(X_final, y)

print("\nFeature Coefficients (Check how much Text Features helped!):")
for feat, coef in zip(FEATURES, final_model.coef_):
    print(f"  {feat:20s}  {coef:+.6f}")

# ── 6. Generate Predictions for Test Sets ─────────────────────────────────────

print("\nGenerating Predictions...")
def predict_positions(bars: pd.DataFrame, headlines: pd.DataFrame) -> pd.Series:
    feats = build_all_features(bars, headlines)[FEATURES]
    X = final_scaler.transform(feats)
    preds = final_model.predict(X)
    positions = make_positions(preds, sensitivity=SENSITIVITY, clip=CLIP)
    return pd.Series(positions, index=feats.index, name="target_position")

public_positions  = predict_positions(public_bars, headline_public)
private_positions = predict_positions(private_bars, headline_private)

# ── 7. Write Submissions ──────────────────────────────────────────────────────

def write_submission(positions: pd.Series, path: str):
    sub = positions.reset_index()
    sub.columns = ["session", "target_position"]
    sub = sub.sort_values("session")
    sub.to_csv(path, index=False)
    print(f"Saved {len(sub)} rows → {path}")

write_submission(public_positions,  "submission_public_9.csv")
write_submission(private_positions, "submission_private_9.csv")

print("\n🚀 DONE! Submissions ready for upload.")

Loading data...
Building features...


/tmp/ipykernel_361193/2289190169.py:48: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return bars.groupby("session").apply(session_feats)
/tmp/ipykernel_361193/2289190169.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return merged.groupby("session").apply(session_text_feats)


Running Cross-Validation...
Baseline CV Sharpe : 2.7838 ± 0.5857
Model    CV Sharpe : 2.6703 ± 0.5965

Training Final Model...

Feature Coefficients (Check how much Text Features helped!):
  cum_return            -0.000111
  last3_return          +0.000086
  slope                 -0.000105
  realized_vol          +0.000121
  range_mean            +0.000135
  up_bar_frac           -0.000091
  wick_ratio            +0.000112
  news_volume           -0.000021
  mean_sentiment        +0.000025
  sum_sentiment         +0.000050
  mean_polarity         +0.000042
  max_impact            +0.000005

Generating Predictions...


/tmp/ipykernel_361193/2289190169.py:48: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return bars.groupby("session").apply(session_feats)
/tmp/ipykernel_361193/2289190169.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return merged.groupby("session").apply(session_text_feats)


In [85]:
import pandas as pd

# 1. Extract coefficients and match them to feature names
coef_series = pd.Series(final_model.coef_, index=FEATURES)

# 2. Take the absolute value, because a strong negative correlation is still highly significant
importance = coef_series.abs().sort_values(ascending=False)

print("--- Feature Importance in Ridge Model ---")
print(importance)

# 3. Quick check: Are the TEXT_FEATURES near the top?
text_importance = importance[importance.index.isin(TEXT_FEATURES)].sum()
price_importance = importance[importance.index.isin(PRICE_FEATURES)].sum()

print(f"\nTotal Weight given to Price: {price_importance:.6f}")
print(f"Total Weight given to Text:  {text_importance:.6f}")

--- Feature Importance in Ridge Model ---
range_mean        0.000044
realized_vol      0.000039
wick_ratio        0.000038
cum_return        0.000037
slope             0.000036
up_bar_frac       0.000032
last3_return      0.000025
sum_sentiment     0.000014
mean_polarity     0.000012
mean_sentiment    0.000007
news_volume       0.000006
max_impact        0.000004
dtype: float64

Total Weight given to Price: 0.000251
Total Weight given to Text:  0.000044


In [94]:
final_submission = pd.concat([public_positions, private_positions])

In [82]:
final_submission.head()

session
1000    1.676566
1001    0.069962
1002    0.665171
1003    0.598091
1004    0.855057
Name: target_position, dtype: float64

In [96]:
write_submission(final_submission, "final_submission_8.csv")

Saved 20000 rows → final_submission_8.csv


In [56]:
import os
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold

# ── 1. Configuration & Loading ────────────────────────────────────────────────
SENSITIVITY = 1.0
CLIP = 1.0
DATA_DIR = "../data" # Make sure this matches your environment

print("Loading data...")
# Load Price Data
seen_train    = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_train.parquet"), engine="fastparquet")
unseen_train  = pd.read_parquet(os.path.join(DATA_DIR, "bars_unseen_train.parquet"), engine="fastparquet")
public_bars   = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_public_test.parquet"), engine="fastparquet")
private_bars  = pd.read_parquet(os.path.join(DATA_DIR, "bars_seen_private_test.parquet"), engine="fastparquet")

# Load Pre-processed Headline Data 
headline_train   = pd.read_csv("headline_stats_seen_train.csv")
headline_public  = pd.read_csv("headline_stats_seen_pubtest.csv")
headline_private = pd.read_csv("headline_stats_seen_privtest.csv")

# ── 2. Feature Engineering (Price & Text) ─────────────────────────────────────

def build_price_features(bars: pd.DataFrame) -> pd.DataFrame:
    """Extract session-level features from OHLC bars."""
    def session_feats(grp):
        grp = grp.sort_values("bar_ix")
        c = grp["close"].values
        o = grp["open"].values
        h = grp["high"].values
        lo = grp["low"].values
        n = len(c)
        br = np.diff(c) / c[:-1] if n > 1 else np.array([0.0])

        return pd.Series({
            "cum_return":   c[-1] / c[0] - 1,
            "last3_return": c[-1] / c[max(0, n - 4)] - 1,
            "slope":        np.polyfit(np.arange(n), c, 1)[0] / c.mean(),
            "realized_vol": np.std(br),
            "range_mean":   np.mean((h - lo) / c),
            "up_bar_frac":  np.mean(c > o),
            "wick_ratio":   np.mean((h - c) / (h - lo + 1e-8)),
        })
    return bars.groupby("session").apply(session_feats)

def build_text_features(headlines: pd.DataFrame, bars: pd.DataFrame) -> pd.DataFrame:
    """Extract session-level features from headline sentiment and price convergence."""
    # Merge bars to get open/close for impact calculation
    merged = headlines.merge(
        bars[['session', 'bar_ix', 'open', 'close']], 
        on=['session', 'bar_ix'], 
        how='left'
    )
    
    # Calculate Impact (Realized Move * Sentiment Polarity)
    merged['bar_return'] = (merged['close'] - merged['open']) / (merged['open'] + 1e-8)
    merged['impact_score'] = merged['bar_return'].abs() * merged['polarity_score'].abs()
    
    def session_text_feats(grp):
        return pd.Series({
            "news_volume":    len(grp),
            "mean_sentiment": grp["linear_score"].mean(),
            "sum_sentiment":  grp["linear_score"].sum(),
            "mean_polarity":  grp["polarity_score"].mean(),
            "max_impact":     grp["impact_score"].max()
        })
        
    return merged.groupby("session").apply(session_text_feats)

def build_all_features(bars: pd.DataFrame, headlines: pd.DataFrame) -> pd.DataFrame:
    """Combines price and text features. Tree models natively handle NaN, but filling 0 for volume is safe."""
    X_price = build_price_features(bars)
    X_text = build_text_features(headlines, bars)
    return X_price.join(X_text, how='left').fillna(0)

# Feature Lists
PRICE_FEATURES = ["cum_return", "last3_return", "slope", "realized_vol", "range_mean", "up_bar_frac", "wick_ratio"]
TEXT_FEATURES  = ["news_volume", "mean_sentiment", "sum_sentiment", "mean_polarity", "max_impact"]
FEATURES = PRICE_FEATURES + TEXT_FEATURES

# ── 3. Build Training Labels & Matrix ─────────────────────────────────────────

print("Building features...")
mid_price = seen_train.groupby("session")["close"].last()
end_price = unseen_train.groupby("session")["close"].last()
y_train   = (end_price / mid_price - 1).rename("return")

X_train_raw = build_all_features(seen_train, headline_train)[FEATURES]

# Align index
idx = X_train_raw.index.intersection(y_train.index)
X_train_raw = X_train_raw.loc[idx]
y_train     = y_train.loc[idx]

# ── 4. Cross-Validation & Position Sizing ─────────────────────────────────────

def compute_sharpe(positions, returns):
    pnl = positions * returns
    return np.mean(pnl) / (np.std(pnl) + 1e-8) * 16

def make_positions(raw_preds, long_bias=1.0, sensitivity=0.3, clip=2.5):
    z = (raw_preds - raw_preds.mean()) / (raw_preds.std() + 1e-8)
    z = np.clip(z, -clip, clip)
    positions = long_bias + sensitivity * z
    positions = np.maximum(positions, 0.05)
    return positions

print("Running Cross-Validation...")
# No scaler needed for LightGBM!
X = X_train_raw.values
y = y_train.values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_sharpes_model = []
cv_sharpes_baseline = []

for train_idx, val_idx in kf.split(X):
    # Initialize LightGBM Regressor
    model = LGBMRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=5,
        colsample_bytree=0.7, # Excellent for preventing overfitting on financial data
        random_state=42,
        verbose=-1 # Keeps the console clean during the CV loop
    )
    model.fit(X[train_idx], y[train_idx])

    preds     = model.predict(X[val_idx])
    positions = make_positions(preds, sensitivity=SENSITIVITY, clip=CLIP)
    y_val     = y[val_idx]

    cv_sharpes_model.append(compute_sharpe(positions, y_val))
    cv_sharpes_baseline.append(compute_sharpe(np.ones(len(y_val)), y_val))

print(f"Baseline CV Sharpe : {np.mean(cv_sharpes_baseline):.4f} ± {np.std(cv_sharpes_baseline):.4f}")
print(f"Model    CV Sharpe : {np.mean(cv_sharpes_model):.4f} ± {np.std(cv_sharpes_model):.4f}")

# ── 5. Train Final Model ──────────────────────────────────────────────────────

print("\nTraining Final Model...")
final_model = LGBMRegressor(
    n_estimators=200,         # Slightly more trees...
    learning_rate=0.03,       # ...learning slightly slower.
    max_depth=3,              # Strict structural limit (the secret sauce!)
    colsample_bytree=0.6,     # Randomize features
    subsample=0.8,            # Randomize rows
    subsample_freq=1,         # Required to enable 'subsample'
    random_state=42,
    verbose=-1
)

# Train directly on the raw data (no scaler)
final_model.fit(X_train_raw, y)

print("\nFeature Importances (Top Drivers of Predictions):")
# LightGBM uses feature importances instead of coefficients
importances = final_model.feature_importances_
feat_imps = sorted(zip(FEATURES, importances), key=lambda x: x[1], reverse=True)

for feat, imp in feat_imps:
    print(f"  {feat:20s}  {imp}")

# ── 6. Generate Predictions for Test Sets ─────────────────────────────────────

print("\nGenerating Predictions...")
def predict_positions(bars: pd.DataFrame, headlines: pd.DataFrame) -> pd.Series:
    feats = build_all_features(bars, headlines)[FEATURES]
    # Feed raw features directly into the tree model
    preds = final_model.predict(feats)
    positions = make_positions(preds, sensitivity=SENSITIVITY, clip=CLIP)
    return pd.Series(positions, index=feats.index, name="target_position")

public_positions  = predict_positions(public_bars, headline_public)
private_positions = predict_positions(private_bars, headline_private)

# ── 7. Write Submissions ──────────────────────────────────────────────────────

def write_submission(positions: pd.Series, path: str):
    sub = positions.reset_index()
    sub.columns = ["session", "target_position"]
    sub = sub.sort_values("session")
    sub.to_csv(path, index=False)
    print(f"Saved {len(sub)} rows → {path}")

write_submission(public_positions,  "submission_public_2.csv")
write_submission(private_positions, "submission_private_2.csv")

print("\n🚀 DONE! Submissions ready for upload.")

Loading data...
Building features...


/tmp/ipykernel_361193/1563037910.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return bars.groupby("session").apply(session_feats)
/tmp/ipykernel_361193/1563037910.py:70: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return merged.groupby("session").apply(session_text_feats)
/home/anna/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature 

Running Cross-Validation...


/home/anna/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/anna/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Baseline CV Sharpe : 2.7838 ± 0.5857
Model    CV Sharpe : 2.6830 ± 0.4561

Training Final Model...

Feature Importances (Top Drivers of Predictions):
  range_mean            152
  slope                 114
  max_impact            94
  realized_vol          93
  last3_return          92
  sum_sentiment         86
  wick_ratio            81
  cum_return            78
  mean_polarity         74
  mean_sentiment        70
  news_volume           56
  up_bar_frac           32

Generating Predictions...


/tmp/ipykernel_361193/1563037910.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return bars.groupby("session").apply(session_feats)
/tmp/ipykernel_361193/1563037910.py:70: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return merged.groupby("session").apply(session_text_feats)
/tmp/ipykernel_361193/1563037910.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behav

Saved 10000 rows → submission_public_2.csv
Saved 10000 rows → submission_private_2.csv

🚀 DONE! Submissions ready for upload.


/tmp/ipykernel_361193/1563037910.py:70: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return merged.groupby("session").apply(session_text_feats)


In [57]:
final_submission2 = pd.concat([public_positions, private_positions])

In [58]:
write_submission(final_submission2, "final_submission2.csv")

Saved 20000 rows → final_submission2.csv
